# Apple Damage Detector - YOLOv5 Training & Export

This notebook trains a `yolov5n` (nano) model for Apple Fruit Damage Detection and exports it to ONNX format (opset 12) with embedded class names for deployment on Raspberry Pi 4.

### Target Classes:
0. `apple_fresh` (HEALTHY)
1. `apple_rotten` (ROT, rotten)
2. `apple_bruised` (BLOTCH, bruising)
3. `apple_damaged` (SCAB, crack, insect_bite)

## 1. Setup Environment

In [ ]:
!git clone -q https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -q -r requirements.txt roboflow onnx onnxsim

## 2. Download Dataset
Set your Roboflow API key in Colab Secrets (named `ROBOFLOW_API_KEY`) or paste it below.

In [ ]:
import os
from google.colab import userdata
from roboflow import Roboflow

try:
    api_key = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    api_key = os.environ.get("ROBOFLOW_API_KEY", "YOUR_API_KEY_HERE")

rf = Roboflow(api_key=api_key)
# Using merged project or downloading dataset
project = rf.workspace("ashwins-workspace-wsaqd").project("apple-fruit-disease-detection-zei6u-ggfei")
version = project.version(1)
dataset = version.download("yolov5")
DATA_YAML = f"{dataset.location}/data.yaml"
print("Downloaded dataset location:", dataset.location)

## 3. Verify Class Names in data.yaml

In [ ]:
import yaml

with open(DATA_YAML, "r") as f:
    data_config = yaml.safe_load(f)

names = data_config.get("names", [])
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]

print("Classes found:", names)
expected = ["apple_fresh", "apple_rotten", "apple_bruised", "apple_damaged"]

# If using raw dataset before remapping, run dataset_prep.py or verify exact mapping
print("Expected classes:", expected)

## 4. Train YOLOv5n Model
- 416x416 input size
- 100 epochs
- Patience 20

In [ ]:
!python train.py --img 416 --batch 16 --epochs 100 --patience 20 \
  --data "{DATA_YAML}" --weights yolov5n.pt --project runs --name apple --exist-ok

## 5. Evaluate Validation Metrics (mAP50 & Per-Class Recall)

In [ ]:
!python val.py --weights runs/apple/weights/best.pt --data "{DATA_YAML}" --img 416 --task val

## 6. Export to ONNX (Opset 12, Simplified) & Embed Metadata

In [ ]:
!python export.py --weights runs/apple/weights/best.pt --include onnx --img 416 --opset 12 --simplify

import ast
import onnx
import onnxruntime as ort

onnx_file = "runs/apple/weights/best.onnx"
model = onnx.load(onnx_file)
meta_dict = {prop.key: prop.value for prop in model.metadata_props}
print("Embedded metadata keys:", list(meta_dict.keys()))

sess = ort.InferenceSession(onnx_file, providers=["CPUExecutionProvider"])
print("ONNX Input shape:", sess.get_inputs()[0].shape)
print("ONNX Model Metadata:", sess.get_modelmeta().custom_metadata_map)

## 7. Download Model Weights for Raspberry Pi Deployment

In [ ]:
from google.colab import files
files.download("runs/apple/weights/best.pt")
files.download("runs/apple/weights/best.onnx")